In [ ]:
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
import torch
import numpy as np 
import matplotlib.pyplot as plt

In [43]:
class SimpleNN(nn.Module):
    def __init__(self, d):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(d * d, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, d),
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [45]:
def setup_device():
    try:
        import torch_directml

        device = torch_directml.device()
        backend = "directml"    
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            backend = "cuda"
        else:
            device = torch.device("cpu")
            backend = "cpu"
    return device

device = setup_device()

In [46]:
classifier = SimpleNN(d=5)
classifier.load_state_dict(torch.load("sandbox/model_full.pth"))

C:\Users\micha\AppData\Local\Temp\ipykernel_3960\2366987388.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  classifier.load_state_dict(torch.load("sandbox/model_full.pth

<All keys matched successfully>

In [49]:
def generate_matrix(d, block_size, mode, eps=None, lam=1, value_range=None, schur=False):
    """
    Generuje macierz testową o rozmiarze `d x d` z kontrolowaną strukturą.

    Parametry:
    - d (int): wymiar macierzy (liczba wierszy/kolumn).
    - block_size (int): liczba pozycji na nadprzekątnej, które zostaną ustawione
        na 1 (losowo wybrane indeksy). Używane do budowy macierzy Jordanowskiej.
    - mode (str): tryb generacji macierzy pomocniczej `S`. Obsługiwane wartości:
        "random", "int", "upper", "lower", "ortho". Określa, jakiego rodzaju
        macierz S zostanie wygenerowana (np. ortogonalna, trójkątna itp.).
    - eps (float|None): jeśli nie None, dodaje małą losową perturbację do macierzy
        J (skaluje losowymi wartościami z [0, eps)). Przydatne do testów stabilności.
    - lam (float): wartość dodawana do elementów diagonalnych macierzy J (domyślnie 1).
    - value_range (int|float|None): skala wartości używana przy generacji macierzy S;
        jeśli None, wartość dobierana jest zależnie od `mode`.
    - schur (bool): jeśli True, zwraca postać Schura macierzy X zamiast X bezpośrednio.

    Uwaga:
    Funkcja tworzy macierz J (prawie Jordanowską) z nadprzekątną ustawioną zgodnie
    z `block_size`, a następnie wykonuje podobieństwo X = S @ J @ S^{-1}, gdzie S
    jest generowane według `mode`.
    """
    indexes = np.random.choice(d-1, size=block_size, replace=False)
    # indexes = list(range(block_size))
    super_diag = np.zeros(d-1)
    for index in indexes:
        super_diag[index] = 1
    J = lam * np.eye(d) + np.diag(super_diag, k=1)
    if eps is not None:
        J += eps * np.random.randn(d, d)
    if value_range is None:
        match mode:
            case "random" | "upper" | "ortho" | "lower":
                value_range = 1
            case "int":
                value_range = 100
            case _:
                raise RuntimeError(f"Mode {mode} is not supported")

    def generate_S():
        while True:
            match mode:
                case "random":
                    S = np.random.randn(d, d) * value_range
                case "int":
                    S = np.random.randint(0, value_range, size=(d, d))
                case "upper":
                    S = np.triu(np.random.randn(d, d)) * value_range
                case "lower":
                    S = np.tril(np.random.randn(d, d)) * value_range
                case "ortho":
                    A = np.random.randn(d, d)
                    Q, _ = np.linalg.qr(A)
                    S = Q
            if abs(np.linalg.cond(S)) < 1e5:
                return S

    S = generate_S()
    return S, J

In [52]:
S_true, J_true = generate_matrix(5, 3, "random")
X = np.linalg.inv(S_true) @ J_true @ S_true

In [53]:
print(J_true)

[[1. 0. 0. 0. 0.]
 [0. 1. 1. 0. 0.]
 [0. 0. 1. 1. 0.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]]


In [36]:
S = torch.randn(5,5, dtype=float, requires_grad=True)
X = torch.tensor(X)
J = torch.tensor(J_true)

# Optimizer
optimizer = torch.optim.Adam([S], lr=1e-2)

tol = 1e-3

# Optimization loop
num_steps = 10000
for step in range(num_steps):
    optimizer.zero_grad()

    # f(S) = || S @ X - J @ S ||_F
    loss = torch.norm(S @ X - J @ S, p='fro')

    if loss < tol:
        print(f"Step {step}: loss = {loss.item():.6f} < tol = {tol:.6f}. Breaking...")
        break

    # Backprop
    loss.backward()

    # Gradient step
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}: loss = {loss.item():.6f}")


Step 0: loss = 4.835027
Step 100: loss = 0.080433
Step 200: loss = 0.011558
Step 300: loss = 0.014354
Step 400: loss = 0.006431
Step 500: loss = 0.011891
Step 600: loss = 0.006083
Step 700: loss = 0.004637
Step 736: loss = 0.000603 < tol = 0.001000. Breaking...


In [37]:
S_found = S.detach().numpy()

In [38]:
print(X.numpy())

[[ 1.14333009 -1.45380724  0.2519596   2.18939121  1.03983265]
 [ 0.04273351  0.56655102  0.07512112  0.65276149  0.3100235 ]
 [-0.02635181  0.26728823  0.95367623 -0.40252827 -0.19117736]
 [ 0.01595291 -0.1618115   0.02804358  1.24368339  0.11573535]
 [ 0.0127859  -0.12968827  0.0224763   0.19530675  1.09275927]]


In [39]:
X_rec = np.linalg.inv(S_found) @ J.numpy() @ S_found
print(np.linalg.norm(X - X_rec))

0.04011809109784786


C:\Users\micha\AppData\Local\Temp\ipykernel_3960\3330244725.py:2: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  print(np.linalg.norm(X - X_rec))
